# Act I — The Shared Boom & The Price Divergence
### *The Diverging American Dream: Can America Afford a Family?*

**Story beat:** From 1960 to ~1973, the American economy delivered broadly shared prosperity.
Real median family income nearly doubled. A single earner could buy a median home for
roughly 3× their annual income. Fertility was above replacement. Then something changed.

This notebook builds **4 static visualizations** that together establish the "before" picture
and introduce the price divergence that defines the entire project.

| # | Visualization | Type | Data |
|---|---|---|---|
| 1 | The Two Inflations — CPI Category Divergence | Static multi-line | FRED CPI series |
| 2 | The Shared Boom — Real Median Family Income | Static area + line | MEFAINUSA672N |
| 3 | Fertility Was Above Replacement | Static annotated line | SPDYNTFRTINUSA |
| 4 | A Home Cost 3× Your Income, Not 5× | Static dual-axis | MSPUS + MEFAINUSA672N |

---


## Setup — Imports, Theme, Data Loading

In [19]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ── Import shared theme ──────────────────────────────────────
from theme import (
    THEME, C,
    apply_theme, apply_dark_theme,
    annotate_threshold, add_recession_bands, watermark,
    apply_mpl_style, PLOTLY_LAYOUT
)

# ── Paths ────────────────────────────────────────────────────
RAW = "/Users/ishika/Desktop/MS/Scholarship_Project/Data/"          # adjust if your folder is different
# If running from same folder as data/raw, use:
# RAW = "data/raw/"

print("✓ Imports complete")
print(f"  Theme background : {C.bg}")
print(f"  Red (rising costs): {C.red}")
print(f"  Blue (wages)      : {C.blue}")
print(f"  Gold (reference)  : {C.gold}")
print(f"  Green (fell)      : {C.green}")


✓ Imports complete
  Theme background : #F5F0E8
  Red (rising costs): #B84A2E
  Blue (wages)      : #2A6FAF
  Gold (reference)  : #C8872A
  Green (fell)      : #2A7C6F


## Load & Prepare Raw FRED Data

In [20]:
def load_fred(filename, col_rename=None, start_year=1960, end_year=2025):
    """
    Load a FRED CSV.  Column 1 = DATE, Column 2 = series value.
    Returns a pandas Series indexed by integer year, annual frequency.
    """
    df = pd.read_csv(RAW + filename)

    # Parse M/D/YY strings explicitly and fix two-digit-year rollover (e.g., 1/1/60 -> 1960, not 2060).
    df['DATE'] = pd.to_datetime(df['DATE'], format='%m/%d/%y', errors='coerce')
    rollover = df['DATE'].dt.year > end_year
    if rollover.any():
        df.loc[rollover, 'DATE'] = df.loc[rollover, 'DATE'] - pd.DateOffset(years=100)

    df['year'] = df['DATE'].dt.year
    val_col = [c for c in df.columns if c not in ['DATE', 'year']][0]
    df = df.rename(columns={val_col: col_rename or val_col})
    name = col_rename or val_col
    df = df.dropna(subset=[name])
    df = df[(df['year'] >= start_year) & (df['year'] <= end_year)]
    # Annual: take mean if multiple obs per year (monthly/quarterly -> annual)
    annual = df.groupby('year')[name].mean()
    return annual


def index_to_100(series, base_year):
    """Re-index a series so base_year == 100. Returns rounded Series."""
    base = series.loc[base_year] if base_year in series.index else series.dropna().iloc[0]
    return (series / base * 100).round(2)


# ── Load all Act 1 series ────────────────────────────────────
BASE_YEAR = 1967   # earliest year with CPIFABSL + CUUR0000SEMC both available

# Costs that ROSE (red / orange / purple family)
s_shelter   = load_fred('CUSR0000SAH1.csv',    'shelter')
s_medical   = load_fred('CPIMEDSL.csv',         'medical_care')
s_hospital  = load_fred('CUUR0000SEMC.csv',     'hospital_services')
s_tuition   = load_fred('CUUR0000SEEB.csv',     'tuition_childcare')

# Costs that FELL or kept pace (green family)
s_apparel   = load_fred('CPIAPPSL.csv',         'apparel')
s_food      = load_fred('CPIFABSL.csv',         'food_beverages')
s_transport = load_fred('CPITRNSL.csv',         'transportation')

# Neutral baselines
s_cpi_all   = load_fred('CPIAUCSL.csv',         'all_items_cpi')
s_wages_nom = load_fred('AHETPI.csv',           'avg_hourly_earnings')

# Income, housing, fertility
s_income    = load_fred('MEFAINUSA672N.csv',    'real_median_family_income', start_year=1953)
s_tfr       = load_fred('SPDYNTFRTINUSA.csv',   'total_fertility_rate', start_year=1960)
s_home      = load_fred('MSPUS.csv',            'median_home_price',    start_year=1963)
s_mortgage  = load_fred('MORTGAGE30US.csv',     'mortgage_rate_30yr',   start_year=1971)
s_comp      = load_fred('COMPRNFB.csv',         'real_compensation',    start_year=1960)

# ── Index all CPI series to BASE_YEAR = 100 ──────────────────
idx = {
    'shelter':           index_to_100(s_shelter,   BASE_YEAR),
    'medical_care':      index_to_100(s_medical,   BASE_YEAR),
    'hospital_services': index_to_100(s_hospital,  BASE_YEAR),
    'tuition_childcare': index_to_100(s_tuition,   1978),   # starts 1978, index there
    'apparel':           index_to_100(s_apparel,   BASE_YEAR),
    'food_beverages':    index_to_100(s_food,      BASE_YEAR),
    'transportation':    index_to_100(s_transport, BASE_YEAR),
    'all_items_cpi':     index_to_100(s_cpi_all,   BASE_YEAR),
    'avg_hourly_earnings': index_to_100(s_wages_nom, BASE_YEAR),
}

# ── Nominal → real price-to-income ratio ─────────────────────
# Both in nominal terms: nominal home price / nominal family income
# Real family income is in 2023$, deflate back to nominal using CPI
cpi_2023 = s_cpi_all.loc[2023]
mfi_nominal = s_income * (s_cpi_all / cpi_2023)
price_to_income = (s_home / mfi_nominal).round(2)

print("✓ All series loaded")
print(f"\nYear ranges:")
for name, s in [
    ('Shelter CPI',         s_shelter),
    ('Medical CPI',         s_medical),
    ('Hospital CPI',        s_hospital),
    ('Tuition+Childcare',   s_tuition),
    ('Apparel CPI',         s_apparel),
    ('Food CPI',            s_food),
    ('Transport CPI',       s_transport),
    ('Avg Hourly Earnings', s_wages_nom),
    ('Real Median Income',  s_income),
    ('Fertility Rate',      s_tfr),
    ('Median Home Price',   s_home),
]:
    print(f"  {name:22s}: {int(s.index.min())}–{int(s.index.max())}  "
          f"({len(s)} obs)")

print(f"\nPrice-to-income ratio:")
for yr in [1963, 1967, 1970, 1973, 1980, 1990, 2000, 2010, 2020, 2022]:
    if yr in price_to_income.index and not pd.isna(price_to_income[yr]):
        print(f"  {yr}: {price_to_income[yr]:.2f}×")


✓ All series loaded

Year ranges:
  Shelter CPI           : 1960–2025  (66 obs)
  Medical CPI           : 1960–2025  (66 obs)
  Hospital CPI          : 1967–2025  (59 obs)
  Tuition+Childcare     : 1978–2025  (48 obs)
  Apparel CPI           : 1960–2025  (66 obs)
  Food CPI              : 1967–2025  (59 obs)
  Transport CPI         : 1960–2025  (66 obs)
  Avg Hourly Earnings   : 1964–2025  (62 obs)
  Real Median Income    : 1953–2024  (72 obs)
  Fertility Rate        : 1960–2024  (65 obs)
  Median Home Price     : 1963–2025  (63 obs)

Price-to-income ratio:
  1963: 3.43×
  1967: 3.41×
  1970: 2.76×
  1973: 3.14×
  1980: 3.35×
  1990: 3.69×
  2000: 3.42×
  2010: 3.72×
  2020: 3.82×
  2022: 4.56×


---
## Visualization 1 — "The Two Inflations"
**Static multi-line chart**

The hero chart of the entire project. All CPI categories indexed to 100 at 1967.
Red lines = costs that rose faster than wages (the family-formation squeeze).
Green lines = costs that fell (consumer goods).
The thick dark line = average hourly earnings — the worker's yardstick.

> **The "aha"**: A general audience can see in a single image that family-formation
> costs and consumer-electronics costs went in opposite directions — two entirely
> different economies stacked on the same paycheck.


In [21]:
# ── Series config: (label, color, dash, width, annotation_side) ────────────
SERIES = [
    # Rising costs — red/orange/purple
    ('hospital_services', 'Hospital Services',       C.red,        'solid',  3.0, 'right'),
    ('medical_care',      'Medical Care',             C.orange,     'solid',  2.5, 'right'),
    ('shelter',           'Housing',        C.gold,       'solid',  2.5, 'right'),
    ('tuition_childcare', 'Tuition & Childcare*',    C.purple,     'solid',  2.5, 'right'),
    # Neutral reference
    ('all_items_cpi',     'All Items CPI',            C.gray,       'dash',   1.8, 'right'),
    ('avg_hourly_earnings','Avg. Hourly Earnings',    C.blue,       'dot',    2.8, 'right'),
    # Falling costs — green
    ('food_beverages',    'Food & Beverages',         C.green,      'solid',  2.0, 'right'),
    ('transportation',    'Transportation',           C.green_light,'solid',  1.8, 'right'),
    ('apparel',           'Apparel',                  '#27ae60',    'solid',  2.0, 'right'),
]

fig = go.Figure()

# ── Shaded region: "family-formation basket" vs "discretionary basket" ──────
# Subtle shading to emphasize the divergence zone post-1979
years_shade = list(range(1979, 2026))
fig.add_vrect(
    x0=1979, x1=2025,
    fillcolor='rgba(184,74,46,0.03)',
    line_width=0,
    layer='below',
)

# ── Add each series ──────────────────────────────────────────
for key, label, color, dash, width, _ in SERIES:
    s = idx[key].dropna()

    # For tuition: note it starts 1978, so add a marker at first point
    first_year = int(s.index.min())

    fig.add_trace(go.Scatter(
        x=s.index.tolist(),
        y=s.values.tolist(),
        name=label,
        mode='lines',
        line=dict(color=color, width=width, dash=dash),
        hovertemplate=(
            f'<b>{label}</b><br>'
            'Year: %{x}<br>'
            'Index: %{y:.1f}<br>'
            '<extra></extra>'
        ),
    ))

# ── Vertical annotation: 1979 structural break ───────────────
fig.add_vline(
    x=1979,
    line_dash='dot',
    line_color=C.muted,
    line_width=1.2,
    annotation_text='1979 — Structural break',
    annotation_position='top right',
    annotation_font=dict(size=10, color=C.muted, family=THEME['font_mono']),
)

# ── Add recession bands (subtle gray) ───────────────────────
add_recession_bands(fig)

# ── Layout ───────────────────────────────────────────────────
apply_theme(fig)

fig.update_layout(
    title=dict(
        text='<b>The Two Inflations</b><br>'
             '<sup>All categories indexed to 100 in 1967 · '
             'Shaded recessions · *Tuition & Childcare indexed to 1978</sup>',
        font=dict(size=THEME['font_size']['lg'], family=THEME['font']),
    ),
    xaxis=dict(
        title='Year',
        range=[1960, 2025],
        tickmode='linear', dtick=5,
        showgrid=True,
    ),
    yaxis=dict(
        title='Price Index',
        range=[0, 2500],
        showgrid=True,
    ),
    legend=dict(
        orientation='v',
        x=1.01, y=1,
        xanchor='left',
        font=dict(size=11),
    ),
    height=580,
    margin=dict(t=100, b=110, l=70, r=220),
)

fig.show()

# ── Print the key numbers for annotation ────────────────────
print("\n── Key index values in 2024/2025 (latest available) ──")
for key, label, *_ in SERIES:
    s = idx[key].dropna()
    last_yr = int(s.index.max())
    last_val = float(s.iloc[-1])
    base_note = '(1967=100)' if key != 'tuition_childcare' else '(1978=100)'
    print(f"  {label:30s}: {last_val:7.1f} in {last_yr}  {base_note}")



── Key index values in 2024/2025 (latest available) ──
  Hospital Services             :  1421.6 in 2025  (1967=100)
  Medical Care                  :  2060.7 in 2025  (1967=100)
  Housing                       :  1444.0 in 2025  (1967=100)
  Tuition & Childcare*          :  1489.5 in 2025  (1978=100)
  All Items CPI                 :   964.7 in 2025  (1967=100)
  Avg. Hourly Earnings          :  1095.8 in 2025  (1967=100)
  Food & Beverages              :   962.0 in 2025  (1967=100)
  Transportation                :   818.3 in 2025  (1967=100)
  Apparel                       :   257.8 in 2025  (1967=100)


### Key Findings from Visualization 1

The numbers speak for themselves. Since 1967:


In [22]:
# Build a clean summary table of the divergence
rows = []
for key, label, *_ in SERIES:
    s = idx[key].dropna()
    base_yr = 1978 if key == 'tuition_childcare' else BASE_YEAR
    last_yr = int(s.index.max())
    last_val = float(s.iloc[-1])
    # Growth relative to wages
    wage_val = float(idx['avg_hourly_earnings'].loc[last_yr]) if last_yr in idx['avg_hourly_earnings'].index else None

    rows.append({
        'Category': label,
        f'Index Value ({last_yr})': f'{last_val:.0f}',
        'Base Year': base_yr,
        'vs. Wages': f'{last_val/wage_val*100:.0f}% of wage growth' if wage_val else '—',
        'Direction': '📈 Rose faster than wages' if last_val > (wage_val or 999) else '📉 Kept pace or fell',
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

print()
print("── Interpretation ──────────────────────────────────────────────────────")
hosp = float(idx['hospital_services'].dropna().iloc[-1])
app  = float(idx['apparel'].dropna().iloc[-1])
wage = float(idx['avg_hourly_earnings'].dropna().iloc[-1])
tuit = float(idx['tuition_childcare'].dropna().iloc[-1])
print(f"  Hospital services : {hosp:.0f}  (wages: {wage:.0f}) → {hosp/wage:.1f}× faster than wage growth")
print(f"  Tuition+Childcare : {tuit:.0f}  (from 1978 base = 100) → {tuit:.0f}% of 1978 price")
print(f"  Apparel           : {app:.0f}   → clothing got {100-app:.0f}% CHEAPER relative to 1967")
print(f"  Wages             : {wage:.0f}  → set as the reference line")


            Category Index Value (2025)  Base Year           vs. Wages                Direction
   Hospital Services               1422       1967 130% of wage growth 📈 Rose faster than wages
        Medical Care               2061       1967 188% of wage growth 📈 Rose faster than wages
             Housing               1444       1967 132% of wage growth 📈 Rose faster than wages
Tuition & Childcare*               1489       1978 136% of wage growth 📈 Rose faster than wages
       All Items CPI                965       1967  88% of wage growth      📉 Kept pace or fell
Avg. Hourly Earnings               1096       1967 100% of wage growth      📉 Kept pace or fell
    Food & Beverages                962       1967  88% of wage growth      📉 Kept pace or fell
      Transportation                818       1967  75% of wage growth      📉 Kept pace or fell
             Apparel                258       1967  24% of wage growth      📉 Kept pace or fell

── Interpretation ─────────────────────

---
## Visualization 2 — "The Shared Boom Was Real"
**Static annotated area chart**

Before we show what broke, we show what worked.
Real median family income (in 2023 dollars) from 1953 to 2024.

The "before" picture: income grew robustly and broadly through 1973.
Then the growth rate slowed dramatically — setting up the divergence story.

> **Key annotation**: Income nearly doubled between 1953 and 1973 (roughly 20 years).
> It took another **50 years** to double again — and that second doubling happened
> almost entirely for top earners, not the median family.


In [27]:
fig2 = go.Figure()

years = s_income.index.tolist()
vals  = s_income.values.tolist()

# ── Shade the "Shared Boom" era (1953–1973) ──────────────────
fig2.add_vrect(
    x0=1953, x1=1973,
    fillcolor=C.blue_fill,
    line_width=0,
    layer='below',
    annotation_text='The Shared Boom (1953–1973)',
    annotation_position='top left',
    annotation_font=dict(size=11, color=C.blue, family=THEME['font_mono']),
)

# ── Area fill under the full income line ─────────────────────
fig2.add_trace(go.Scatter(
    x=years, y=vals,
    fill='tozeroy',
    fillcolor=C.gold_fill,
    line=dict(color=C.gold, width=0),
    showlegend=False,
    hoverinfo='skip',
))

# ── Main income line ─────────────────────────────────────────
fig2.add_trace(go.Scatter(
    x=years,
    y=vals,
    mode='lines',
    name='Real Median Family Income (2023 $)',
    line=dict(color=C.gold, width=3),
    hovertemplate='<b>%{x}</b><br>$%{y:,.0f}<extra></extra>',
))

# ── Recession shading ────────────────────────────────────────
add_recession_bands(fig2)

# ── Annotate key turning points ──────────────────────────────
annotations = [
    dict(x=1953, y=s_income.loc[1953], text='$40,870', yshift=15),
    dict(x=1973, y=s_income.loc[1973], text=f'${s_income.loc[1973]:,.0f}<br><i>+{((s_income.loc[1973]/s_income.loc[1953])-1)*100:.0f}% in 20 yrs</i>', yshift=15),
    dict(x=2023, y=s_income.loc[2023], text=f'${s_income.loc[2023]:,.0f}<br><i>2023</i>', yshift=15),
]
for ann in annotations:
    fig2.add_annotation(
        x=ann['x'], y=ann['y'],
        text=ann['text'],
        showarrow=True, arrowhead=2, arrowcolor=C.gold,
        arrowwidth=1.5,
        font=dict(size=11, color=C.text, family=THEME['font_mono']),
        bgcolor=THEME['background'],
        bordercolor=C.gold,
        borderwidth=1,
        yshift=ann.get('yshift', 0),
        ax=0, ay=-40,
    )

# ── Growth rate annotation ────────────────────────────────────
boom_growth = ((s_income.loc[1973] / s_income.loc[1953]) - 1) * 100
post_growth = ((s_income.loc[2023] / s_income.loc[1973]) - 1) * 100
fig2.add_annotation(
    x=1963, y=55000,
    text=f'<b>+{boom_growth:.0f}%</b><br>in 20 years',
    showarrow=False,
    font=dict(size=14, color=C.blue, family=THEME['font']),
    bgcolor=THEME['background'],
)
fig2.add_annotation(
    x=1998, y=58000,
    text=f'<b>+{post_growth:.0f}%</b><br>in 50 years',
    showarrow=False,
    font=dict(size=14, color=C.muted, family=THEME['font']),
    bgcolor=THEME['background'],
)

apply_theme(fig2)

fig2.update_xaxes(gridcolor='#EDE8DC', gridwidth=0.5)
fig2.update_yaxes(gridcolor='#EDE8DC', gridwidth=0.5)

fig2.update_layout(
    title=dict(
        text='<b>The Shared Boom Was Real</b><br>'
             '<sup>Real Median Family Income, 1953–2024</sup>',
        font=dict(size=THEME['font_size']['lg'], family=THEME['font']),
    ),
    xaxis=dict(title='Year', range=[1953, 2025], tickmode='linear', dtick=10),
    yaxis=dict(
        title='Real Median Family Income',
        tickformat='$,.0f',
        range=[30000, 120000],
    ),
    showlegend=False,
    height=480,
    margin=dict(t=100, b=70, l=90, r=40),
)

fig2.show()

print(f"\n── Key growth rates ──────────────────────────────────────────────────")
print(f"  Shared Boom  (1953–1973): +{boom_growth:.1f}% real income growth in 20 years")
print(f"  Post-Boom    (1973–2023): +{post_growth:.1f}% real income growth in 50 years")
print(f"  Annual pace  (1953–1973): +{(((s_income.loc[1973]/s_income.loc[1953])**(1/20))-1)*100:.2f}%/yr")
print(f"  Annual pace  (1973–2023): +{(((s_income.loc[2023]/s_income.loc[1973])**(1/50))-1)*100:.2f}%/yr")



── Key growth rates ──────────────────────────────────────────────────
  Shared Boom  (1953–1973): +74.3% real income growth in 20 years
  Post-Boom    (1973–2023): +45.1% real income growth in 50 years
  Annual pace  (1953–1973): +2.82%/yr
  Annual pace  (1973–2023): +0.75%/yr


---
## Visualization 3 — "Fertility Was Above Replacement"
**Static annotated time-series**

The U.S. total fertility rate (TFR) — births per woman — from 1960 to 2024.

Act 1's story: fertility starts high (Baby Boom peak 3.65 in 1960), falls naturally
toward replacement as women entered the workforce and the pill became available,
and crosses below replacement in 1972. But it *recovers* back to near-replacement
by the late 1980s. The crisis comes later — after 2007 — when housing costs explode
again. That fall is Act 3's story.

> **Important methodological note**: The TFR is a *period* measure — it captures
> what would happen if women at each age in a given year maintained that rate their
> whole lives. It overstates fertility decline if births are merely *delayed* rather
> than *foregone*. We flag this honestly in the chart.


In [15]:
fig3 = go.Figure()

years_tfr = s_tfr.index.tolist()
vals_tfr  = s_tfr.values.tolist()

# ── Shaded area: above vs below replacement ──────────────────
# Fill red below 2.1 (alarm), fill blue above (healthy)
REPLACEMENT = 2.1

above_y = [v if v >= REPLACEMENT else REPLACEMENT for v in vals_tfr]
below_y = [v if v <= REPLACEMENT else REPLACEMENT for v in vals_tfr]

# Area above replacement
fig3.add_trace(go.Scatter(
    x=years_tfr + years_tfr[::-1],
    y=above_y + [REPLACEMENT]*len(years_tfr),
    fill='toself',
    fillcolor=C.blue_fill,
    line=dict(width=0),
    showlegend=False, hoverinfo='skip',
))

# Area below replacement (becomes more prominent post-1972)
fig3.add_trace(go.Scatter(
    x=years_tfr + years_tfr[::-1],
    y=below_y + [REPLACEMENT]*len(years_tfr),
    fill='toself',
    fillcolor=C.red_fill,
    line=dict(width=0),
    showlegend=False, hoverinfo='skip',
))

# ── Main TFR line ─────────────────────────────────────────────
fig3.add_trace(go.Scatter(
    x=years_tfr,
    y=vals_tfr,
    mode='lines',
    name='Total Fertility Rate',
    line=dict(color=C.gold, width=3.2),
    hovertemplate='<b>%{x}</b><br>TFR: %{y:.3f}<extra></extra>',
))

# ── Replacement rate reference line ──────────────────────────
annotate_threshold(fig3, REPLACEMENT,
                   'Replacement level (2.1)',
                   color=C.red, dash='dash')

# ── Key event annotations ─────────────────────────────────────
key_events = [
    (1960, 3.654, 'Baby Boom peak<br><b>3.65</b>', -50, 15),
    (1972, 2.010, 'First dip below<br>replacement<br><b>2.01</b>', 20, -55),
    (1976, 1.738, 'Post-pill trough<br><b>1.74</b>', 20, -55),
    (2007, 2.120, 'Pre-recession<br>peak <b>2.12</b>', -20, -55),
    (2024, float(s_tfr.iloc[-1]), f'Record low<br><b>{float(s_tfr.iloc[-1]):.2f}</b>', 20, -40),
]

for yr, val, text, ax, ay in key_events:
    if yr in s_tfr.index:
        fig3.add_annotation(
            x=yr, y=val,
            text=text,
            showarrow=True, arrowhead=2,
            arrowcolor=C.red if val < REPLACEMENT else C.blue,
            arrowwidth=1.5,
            font=dict(size=10, color=C.text, family=THEME['font_mono']),
            bgcolor=THEME['background'],
            bordercolor=C.red if val < REPLACEMENT else C.blue,
            borderwidth=1,
            ax=ax, ay=ay,
        )

# ── Shade the post-2007 crisis period ────────────────────────
fig3.add_vrect(
    x0=2007, x1=2024,
    fillcolor='rgba(184,74,46,0.04)',
    line_width=0,
    layer='below',
    annotation_text='Post-2008 decline →',
    annotation_position='top left',
    annotation_font=dict(size=10, color=C.red, family=THEME['font_mono']),
)

add_recession_bands(fig3)
apply_theme(fig3)

fig3.update_layout(
    title=dict(
        text='<b>U.S. Total Fertility Rate, 1960–2024</b><br>'
             '<sup>Births per woman · Blue = above replacement · '
             'Red = below replacement · Dashed line = 2.1 replacement level</sup>',
        font=dict(size=THEME['font_size']['lg'], family=THEME['font']),
    ),
    xaxis=dict(title='Year', range=[1960, 2025], tickmode='linear', dtick=5),
    yaxis=dict(
        title='Total Fertility Rate (births per woman)',
        range=[1.3, 4.0],
        tickformat='.2f',
    ),
    showlegend=False,
    height=480,
    margin=dict(t=100, b=70, l=70, r=60),
)

watermark(fig3, 'Source: World Bank / CDC via FRED · SPDYNTFRTINUSA · '
                'Note: TFR is a period measure; see methodological note above')
fig3.show()

print("\n── Key TFR values ────────────────────────────────────────────────────")
for yr in [1960, 1964, 1972, 1976, 1990, 2000, 2007, 2010, 2020, 2024]:
    if yr in s_tfr.index:
        below = ' ← BELOW REPLACEMENT' if s_tfr[yr] < 2.1 else ''
        print(f"  {yr}: {s_tfr[yr]:.3f}{below}")



── Key TFR values ────────────────────────────────────────────────────
  1960: 3.654
  1964: 3.190
  1972: 2.010 ← BELOW REPLACEMENT
  1976: 1.738 ← BELOW REPLACEMENT
  1990: 2.081 ← BELOW REPLACEMENT
  2000: 2.056 ← BELOW REPLACEMENT
  2007: 2.120
  2010: 1.931 ← BELOW REPLACEMENT
  2020: 1.641 ← BELOW REPLACEMENT
  2024: 1.627 ← BELOW REPLACEMENT


---
## Visualization 4 — "A Home Cost 3× Your Income, Not 5×"
**Static dual-panel chart: Price-to-income ratio + Mortgage rate**

In 1963 the median home cost roughly **3.4× the median family's annual income**.
Today it costs **4–5×** or more — and that's before accounting for the collapse
and recovery of mortgage rates, which further affect monthly payment affordability.

This chart shows **two dimensions of housing affordability**:
- Panel 1: The price-to-income ratio (the structural long-run affordability)
- Panel 2: The 30-year mortgage rate (the monthly payment affordability)

> **Act 1 lesson**: Housing was more expensive than naive accounts suggest even in 1963,
> but the ratio was stable through the 1960s and rose only modestly through 1980.
> The dramatic deterioration comes later — making this a setup for Act 3.


In [16]:
fig4 = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=(
        'Median Home Price ÷ Median Family Income  (Nominal / Nominal)',
        '30-Year Fixed Mortgage Rate (%)',
    ),
    vertical_spacing=0.08,
    row_heights=[0.62, 0.38],
)

# ── Panel 1: Price-to-income ratio ───────────────────────────
pti_clean = price_to_income.dropna()
years_pti = pti_clean.index.tolist()
vals_pti  = pti_clean.values.tolist()

# Fill under the line
fig4.add_trace(go.Scatter(
    x=years_pti, y=vals_pti,
    fill='tozeroy',
    fillcolor=C.red_fill,
    line=dict(width=0),
    showlegend=False, hoverinfo='skip',
), row=1, col=1)

fig4.add_trace(go.Scatter(
    x=years_pti,
    y=vals_pti,
    mode='lines',
    name='Price-to-Income Ratio',
    line=dict(color=C.red, width=3),
    hovertemplate='<b>%{x}</b><br>Ratio: %{y:.2f}×<extra></extra>',
), row=1, col=1)

# Reference lines: "affordable" zone (≤3×) and "stretched" zone (≥5×)
fig4.add_hline(y=3.0, line_dash='dot', line_color=C.green,
               line_width=1.5,
               annotation_text='3× — historically affordable',
               annotation_position='top right',
               annotation_font=dict(size=9, color=C.green,
                                    family=THEME['font_mono']),
               row=1, col=1)
fig4.add_hline(y=5.0, line_dash='dot', line_color=C.red,
               line_width=1.5,
               annotation_text='5× — stretched',
               annotation_position='top right',
               annotation_font=dict(size=9, color=C.red,
                                    family=THEME['font_mono']),
               row=1, col=1)

# Annotate key years
for yr, label in [(1963,'1963: 3.4×'), (2006,'2006: 4.3×'),
                  (2012,'2012: 3.0×'), (2022,'2022: 4.6×')]:
    if yr in pti_clean.index:
        fig4.add_annotation(
            x=yr, y=pti_clean[yr],
            text=label,
            showarrow=True, arrowhead=2, arrowcolor=C.gold,
            arrowwidth=1.2,
            font=dict(size=9, color=C.text, family=THEME['font_mono']),
            bgcolor=THEME['background'],
            bordercolor=C.gold, borderwidth=1,
            ax=0, ay=-35,
            row=1, col=1,
        )

# ── Panel 2: Mortgage rate ────────────────────────────────────
s_mort_clean = s_mortgage.dropna()
fig4.add_trace(go.Scatter(
    x=s_mort_clean.index.tolist(),
    y=s_mort_clean.values.tolist(),
    mode='lines',
    name='30-yr Mortgage Rate',
    line=dict(color=C.purple, width=2.5),
    hovertemplate='<b>%{x}</b><br>Rate: %{y:.2f}%<extra></extra>',
), row=2, col=1)

# Annotate the 1981 peak
peak_yr = int(s_mort_clean.idxmax())
peak_val = float(s_mort_clean.max())
fig4.add_annotation(
    x=peak_yr, y=peak_val,
    text=f'<b>{peak_val:.1f}%</b><br>1981 peak',
    showarrow=True, arrowhead=2, arrowcolor=C.purple,
    arrowwidth=1.2,
    font=dict(size=9, color=C.purple, family=THEME['font_mono']),
    bgcolor=THEME['background'],
    ax=30, ay=-30,
    row=2, col=1,
)

add_recession_bands(fig4, row=1, col=1)
add_recession_bands(fig4, row=2, col=1)
apply_theme(fig4)

fig4.update_layout(
    title=dict(
        text='<b>Housing Affordability, 1963–2025</b><br>'
             '<sup>Top: Years of median family income needed to buy the median home · '
             'Bottom: 30-year fixed mortgage rate</sup>',
        font=dict(size=THEME['font_size']['lg'], family=THEME['font']),
    ),
    height=580,
    showlegend=False,
    margin=dict(t=100, b=80, l=70, r=60),
)

fig4.update_yaxes(
    title_text='Price ÷ Income (×)',
    range=[2.0, 5.5],
    tickformat='.1f',
    row=1, col=1,
)
fig4.update_yaxes(
    title_text='Rate (%)',
    tickformat='.1f',
    row=2, col=1,
)
fig4.update_xaxes(
    title_text='Year',
    tickmode='linear', dtick=5,
    row=2, col=1,
)

watermark(fig4, 'Sources: FRED · MSPUS (median home price) · '
                'MEFAINUSA672N (real median family income, deflated to nominal) · '
                'MORTGAGE30US (30-yr fixed rate, Freddie Mac)')
fig4.show()

print("\n── Price-to-income ratio at key years ──────────────────────────────")
for yr in [1963, 1967, 1970, 1973, 1980, 1990, 2000, 2007, 2012, 2020, 2022, 2024]:
    if yr in pti_clean.index:
        flag = ' ← STRETCHED' if pti_clean[yr] >= 5 else (
               ' ← AFFORDABLE' if pti_clean[yr] <= 3 else '')
        print(f"  {yr}: {pti_clean[yr]:.2f}×{flag}")



── Price-to-income ratio at key years ──────────────────────────────
  1963: 3.43×
  1967: 3.41×
  1970: 2.76× ← AFFORDABLE
  1973: 3.14×
  1980: 3.35×
  1990: 3.69×
  2000: 3.42×
  2007: 4.04×
  2012: 3.94×
  2020: 3.82×
  2022: 4.56×
  2024: 3.85×


---
## Act I — Summary of Key Numbers

The table below extracts the headline statistics from all four visualizations.
These numbers will be used as callout cards on the final HTML website.


In [17]:
print("=" * 65)
print("ACT I — KEY NUMBERS FOR WEBSITE CALLOUT CARDS")
print("=" * 65)

# ── CPI divergence headline numbers ─────────────────────────
latest_hosp = float(idx['hospital_services'].dropna().iloc[-1])
latest_tuit = float(idx['tuition_childcare'].dropna().iloc[-1])
latest_appl = float(idx['apparel'].dropna().iloc[-1])
latest_wage = float(idx['avg_hourly_earnings'].dropna().iloc[-1])

print("\n── The Two Inflations (1967 = 100) ──────────────────────────")
print(f"  Hospital Services index  : {latest_hosp:.0f}  (+{latest_hosp-100:.0f}% vs 1967)")
print(f"  Tuition & Childcare index: {latest_tuit:.0f}  (+{latest_tuit-100:.0f}% vs 1978)")
print(f"  Apparel index            : {latest_appl:.0f}   ({latest_appl-100:+.0f}% vs 1967)")
print(f"  Avg Hourly Earnings index: {latest_wage:.0f}  (+{latest_wage-100:.0f}% vs 1967)")
print(f"  → Hospital care rose {latest_hosp/latest_wage:.1f}× faster than wages since 1967")

# ── Income growth ────────────────────────────────────────────
inc_1960 = float(s_income.loc[1960])
inc_1973 = float(s_income.loc[1973])
inc_2023 = float(s_income.loc[2023])
print(f"\n── Real Median Family Income ─────────────────────────────────")
print(f"  1960: ${inc_1960:>10,.0f}")
print(f"  1973: ${inc_1973:>10,.0f}  (+{((inc_1973/inc_1960)-1)*100:.0f}% in 13 yrs)")
print(f"  2023: ${inc_2023:>10,.0f}  (+{((inc_2023/inc_1973)-1)*100:.0f}% in 50 yrs)")

# ── Fertility ─────────────────────────────────────────────────
print(f"\n── Total Fertility Rate ──────────────────────────────────────")
for yr in [1960, 1972, 1980, 2007, 2024]:
    if yr in s_tfr.index:
        below = ' ↓ BELOW REPLACEMENT' if s_tfr[yr] < 2.1 else ''
        print(f"  {yr}: {s_tfr[yr]:.3f}{below}")

# ── Housing ──────────────────────────────────────────────────
print(f"\n── Price-to-Income Ratio ─────────────────────────────────────")
for yr in [1963, 1973, 1990, 2000, 2022]:
    if yr in pti_clean.index:
        print(f"  {yr}: {pti_clean[yr]:.2f}×")

print()
print("=" * 65)
print("All four static visualizations for Act I complete.")
print("Save figures as HTML snippets or PNG for website embedding.")
print("=" * 65)


ACT I — KEY NUMBERS FOR WEBSITE CALLOUT CARDS

── The Two Inflations (1967 = 100) ──────────────────────────
  Hospital Services index  : 1422  (+1322% vs 1967)
  Tuition & Childcare index: 1489  (+1389% vs 1978)
  Apparel index            : 258   (+158% vs 1967)
  Avg Hourly Earnings index: 1096  (+996% vs 1967)
  → Hospital care rose 1.3× faster than wages since 1967

── Real Median Family Income ─────────────────────────────────
  1960: $    48,760
  1973: $    71,240  (+46% in 13 yrs)
  2023: $   103,400  (+45% in 50 yrs)

── Total Fertility Rate ──────────────────────────────────────
  1960: 3.654
  1972: 2.010 ↓ BELOW REPLACEMENT
  1980: 1.839 ↓ BELOW REPLACEMENT
  2007: 2.120
  2024: 1.627 ↓ BELOW REPLACEMENT

── Price-to-Income Ratio ─────────────────────────────────────
  1963: 3.43×
  1973: 3.14×
  1990: 3.69×
  2000: 3.42×
  2022: 4.56×

All four static visualizations for Act I complete.
Save figures as HTML snippets or PNG for website embedding.


---
## Export Figures

Export all four figures as interactive HTML snippets (for embedding in the website)
and as high-resolution PNG (for the static visualization requirement).


In [26]:
import os

# ── Output directory ─────────────────────────────────────────
OUT_DIR = '../outputs/act1/'
os.makedirs(OUT_DIR, exist_ok=True)

figures = {
    'act1_fig1_two_inflations':    fig,
    'act1_fig2_shared_boom':       fig2,
    'act1_fig3_fertility':         fig3,
    'act1_fig4_housing':           fig4,
}

for name, f in figures.items():
    # ── Interactive HTML snippet ──────────────────────────────
    html_path = OUT_DIR + name + '.html'
    f.write_html(
        html_path,
        include_plotlyjs='cdn',    # lightweight: loads plotly from CDN
        full_html=False,           # snippet only — embeds into main HTML
        config={
            'displayModeBar': True,
            'displaylogo': False,
            'modeBarButtonsToRemove': ['lasso2d', 'select2d'],
            'toImageButtonOptions': {
                'format': 'png',
                'filename': name,
                'scale': 2,
            },
        },
    )
    print(f"  ✓ HTML: {html_path}")

    # ── Static PNG (requires kaleido) ────────────────────────
    try:
        png_path = OUT_DIR + name + '.png'
        f.write_image(png_path, width=1200, height=f.layout.height or 500,
                      scale=2)
        print(f"  ✓ PNG : {png_path}")
    except Exception as e:
        print(f"  ⚠ PNG skipped ({e}) — install kaleido: pip install kaleido")

print(f"\n✅  Act I export complete → {OUT_DIR}")
print("\nFiles ready for website embedding:")
for f_name in sorted(os.listdir(OUT_DIR)):
    size = os.path.getsize(OUT_DIR + f_name)
    print(f"   {f_name:45s}  {size/1024:6.1f} KB")


  ✓ HTML: ../outputs/act1/act1_fig1_two_inflations.html
  ✓ PNG : ../outputs/act1/act1_fig1_two_inflations.png
  ✓ HTML: ../outputs/act1/act1_fig2_shared_boom.html
  ✓ PNG : ../outputs/act1/act1_fig2_shared_boom.png
  ✓ HTML: ../outputs/act1/act1_fig3_fertility.html
  ✓ PNG : ../outputs/act1/act1_fig3_fertility.png
  ✓ HTML: ../outputs/act1/act1_fig4_housing.html
  ✓ PNG : ../outputs/act1/act1_fig4_housing.png

✅  Act I export complete → ../outputs/act1/

Files ready for website embedding:
   act1_fig1_two_inflations.html                    20.0 KB
   act1_fig1_two_inflations.png                    312.2 KB
   act1_fig2_shared_boom.html                       14.3 KB
   act1_fig2_shared_boom.png                       178.5 KB
   act1_fig3_fertility.html                         16.8 KB
   act1_fig3_fertility.png                         213.5 KB
   act1_fig4_housing.html                           17.4 KB
   act1_fig4_housing.png                           278.6 KB
